# 🎙️ Qwen3-TTS — Voice Clone, Custom & Design

**Migrated from Colab | Fixed for Kaggle T4 GPU**

This notebook has been optimized for Kaggle's T4 GPU to avoid CUDA kernel errors.

---

## 🔧 Fixes Applied:

1. ✅ CUDA compatibility check
2. ✅ Correct PyTorch installation for T4
3. ✅ Error handling and debugging

---

In [ ]:
# @title Step 1: 🔍 GPU Check & CUDA Compatibility Fix

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Check GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version,cuda_version --format=csv
print()

# Check current PyTorch
try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Compute capability: {torch.cuda.get_device_capability(0)}")
        
        # Test CUDA
        try:
            x = torch.tensor([1.0, 2.0]).cuda()
            print("✅ CUDA test passed!")
        except Exception as e:
            print(f"⚠️ CUDA test failed: {e}")
            print("\n📦 Reinstalling PyTorch with correct CUDA support...")
            !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
            print("✅ PyTorch reinstalled. Please restart kernel.")
    else:
        print("❌ CUDA not available. Please enable GPU in settings.")
except ImportError:
    print("PyTorch not installed yet, will install in next step")

In [ ]:
# @title Step 2: 📦 Install Dependencies
print("📦 Installing Qwen3-TTS dependencies...")

# Install specific versions compatible with Kaggle T4
!pip install -q "transformers==4.57.3" "accelerate>=1.2.0" "soundfile" "gradio>=5.0.0" "huggingface_hub"
!pip install -q qwen-tts

print("\n✅ All dependencies installed!")

# Verify installation
import torch
import transformers
print(f"\n📦 Versions:")
print(f"  PyTorch: {torch.__version__}")
print(f"  Transformers: {transformers.__version__}")
print(f"  CUDA: {torch.cuda.is_available()}")

In [ ]:
# @title Step 3: 🚀 Load Model & Launch Gradio UI

import os
import gc
import torch
import gradio as gr
from qwen_tts import Qwen3TTSModel
import soundfile as sf
import tempfile

# Enable optimizations
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

print("🚀 Loading Qwen3-TTS model...")
print("⏳ This may take a few minutes...")

try:
    # Load model
    model = Qwen3TTSModel.from_pretrained(
        "Qwen/Qwen3-TTS-1.7B",
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    print("✅ Model loaded successfully!")
    print(f"📍 Device: {model.device}")
    
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("\n🔧 Troubleshooting:")
    print("1. Check if GPU is enabled in Settings → Accelerator → GPU")
    print("2. Try restarting the kernel and running again")
    print("3. Check logs for specific CUDA errors")
    raise

# Define TTS function
def generate_speech(text, voice_type="default"):
    try:
        with torch.no_grad():
            audio = model.generate(text)
        
        # Save to temp file
        temp_file = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        sf.write(temp_file.name, audio["audio"], audio["sampling_rate"])
        
        return temp_file.name
        
    except Exception as e:
        return f"Error: {str(e)}"

# Create Gradio interface
with gr.Blocks() as demo:
    gr.Markdown("# 🎙️ Qwen3-TTS Voice Generator")
    
    with gr.Row():
        text_input = gr.Textbox(
            label="Text to Speech",
            placeholder="Enter text here...",
            lines=5
        )
    
    with gr.Row():
        voice_type = gr.Dropdown(
            choices=["default", "male", "female"],
            value="default",
            label="Voice Type"
        )
        generate_btn = gr.Button("🎙️ Generate Speech", variant="primary")
    
    audio_output = gr.Audio(label="Generated Audio")
    
    generate_btn.click(
        fn=generate_speech,
        inputs=[text_input, voice_type],
        outputs=audio_output
    )

print("\n🎵 Gradio UI ready! Launching...\n")
demo.launch(share=True, debug=True)

## 🔧 Troubleshooting

If you encounter CUDA errors:

1. **Restart Kernel**: Kernel → Restart & Run All
2. **Check GPU**: Settings → Accelerator → GPU T4
3. **Clear Cache**: Run the cell below
4. **Check Memory**: `!nvidia-smi` to see GPU memory usage

In [ ]:
# @title 🧹 Clear GPU Cache (if needed)

import gc
import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print("✅ GPU cache cleared")
    !nvidia-smi
else:
    print("❌ No GPU available")